In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from src.pipeline import *
from src.models.neural_models import *
import warnings
from pandas.errors import PerformanceWarning
warnings.filterwarnings("ignore", category=PerformanceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

KeyboardInterrupt: 

# EDA and Visualizations

In [ ]:
data=pd.read_csv('./data/processed/final_dataset_with_indicators.csv')
data.head()

In [ ]:
plt.plot(data['Date'], data['Ucome_fob_ARA']);

In [ ]:
plt.boxplot(data['Ucome_fob_ARA'])

# Model Training Example

In [ ]:
scaled_data = data_scaler('MinMax')

In [ ]:
target_col = scaled_data['Ucome_fob_ARA']

In [ ]:

#feature selection using random forest
scaled_data = feature_selection(scaled_data)

# Update the number of features after feature selection
n_features = scaled_data.shape[1]  # Number of features (price + indicators)

# Split the data into training and testing sets
train_data, test_data = train_test_split(scaled_data)

# Prepare the data for the neural network
window_size = 30
lookahead_value = 10
batch_size = 32



In [ ]:
windowed_train = window_creator(train_data, window_size, lookahead_value, batch_size)

windowed_test = window_creator(test_data, window_size, lookahead_value, batch_size)

# Linear Models Evaluation

In [ ]:
from src.models.linear_models import run_linear_models_experiment

In [ ]:
linear_results_df, linear_artifacts = run_linear_models_experiment(
    data=scaled_data,
    target_col="Ucome_fob_ARA",
    feature_cols=scaled_data.columns.tolist(),
    horizon=1,
    target_mode="change",
    start_date="2023-01-01",
    train_size=0.70,
    val_size=0.15,
)

linear_results_df

In [ ]:
# Optimize hyperparameters
ltsm_params = optimize_hyperparameters(
    train_df=train_data, 
    model_type='lstm',
    lookahead_value=lookahead_value, 
    batch_size=batch_size,
    window_size=window_size,
    n_features=n_features
)

gru_params = optimize_hyperparameters(
    train_df=train_data,
    model_type='gru',
    lookahead_value=lookahead_value,
    batch_size=batch_size,
    window_size=window_size,
    n_features=n_features
)

hybrid_params = optimize_hyperparameters(
    train_df=train_data, 
    model_type='hybrid',
    lookahead_value=lookahead_value, 
    batch_size=batch_size,
    window_size=window_size,
    n_features=n_features
)


In [ ]:
# neural models
ltsm_model = LSTMModel(
    lstm_units=int(ltsm_params['lstm_units']),
    dropout=float(ltsm_params['dropout']),
    learning_rate=float(ltsm_params['learning_rate']),
    window_size=window_size,
    n_features=n_features
)
gru_model = GRUModel(
    gru_units=int(gru_params['gru_units']),
    dropout=float(gru_params['dropout']),
    learning_rate=float(gru_params['learning_rate']),
    window_size=window_size,
    n_features=n_features
)
hybrid_model = HybridGRU_LSTM(
    gru_units=int(hybrid_params['gru_units']),
    lstm_units=int(hybrid_params['lstm_units']),
    dropout=float(hybrid_params['dropout']),
    learning_rate=float(hybrid_params['learning_rate']),
    window_size=window_size,
    n_features=n_features
)

In [ ]:
print("Training models on training set...")

history = [ltsm_model.train(windowed_train), gru_model.train(windowed_train), hybrid_model.train(windowed_train)]  # train_data contains (X, y) pairs internally

print("Training complete!\n")
 
print("Evaluating model on test set...")
test_metrics = [ltsm_model.evaluate(windowed_test), gru_model.evaluate(windowed_test), hybrid_model.evaluate(windowed_test)]  # test_data contains (X, y) pairs internally
test_metrics_df = pd.DataFrame([test_metrics], columns=['Loss', 'MSE', 'MAE', 'MAPE'], index=['LSTM', 'GRU', 'Hybrid'])
print(test_metrics_df)
print("Generating predictions on test set...")
predictions = [ltsm_model.predict(windowed_test), gru_model.predict(windowed_test), hybrid_model.predict(windowed_test)]
# print(f"Predictions shape: {predictions[0].shape}")
# print(f"First 10 predictions:\n{predictions[0][:10].flatten()}")
# print(f"Prediction range: [{predictions[0].min():.4f}, {predictions[0].max():.4f}]")

print("\nModel training and evaluation complete!")